[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: despliegue en Linux por SSH

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

**Estado de alineación.** Esta solución se escribió contra el contenido de `main` (`dff2150`). La rama de la integración continua `ci/cuadernos-16-17-2711` (`fe1cc48`) solo cambió metadatos: el cuaderno 17 no cambió. **Pendiente de alineación** si esa rama cambia después celdas o *workflows*; las comprobaciones de este cuaderno leen los *workflows* reales y fallarán de forma visible si cambian los nombres de pasos o de variables.

**Requisitos para las celdas con etiqueta.** Un *Ubuntu* 24.04 completo (equipo propio o máquina virtual) con *systemd*, al que se llega por *SSH* con una cuenta administradora; este es el único ejercicio del curso que no cabe en el *Codespace*. Las direcciones, usuarios y puertos se leen de variables de entorno. El ejemplo usa `192.0.2.10`, una dirección reservada para documentación.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Preparar el destino y el runner (o usar un runner hospedado)

En el *Ubuntu* 24.04 propio, preparar el destino con `preparar_anfitrion.sh` y, si no es alcanzable desde internet, registrar ahí un *runner* autoalojado (usuario distinto de `deploy`, fuera del grupo `docker`, repositorio **privado**). Si se usa la alternativa, definir `DEPLOY_RUNNER` como `ubuntu-latest`.

La siguiente celda generará, en una carpeta temporal, un par de claves *ed25519* sin frase de paso, como en el cuaderno original, y comprobará el formato de la clave pública y de su huella. Es un ensayo: la clave real se genera en la celda siguiente, fuera del repositorio.

In [ ]:
import subprocess

TEMPORAL = u.directorio_temporal()
clave = TEMPORAL / "deploy"
subprocess.run(["ssh-keygen", "-q", "-t", "ed25519", "-f", str(clave), "-N", "", "-C", "clave-de-ejemplo"], check=True)

publica = clave.with_suffix(".pub").read_text().split()
huella = subprocess.run(["ssh-keygen", "-lf", str(clave.with_suffix(".pub"))], capture_output=True, text=True, check=True).stdout
print(huella.strip())
assert publica[0] == "ssh-ed25519"
assert huella.split()[1].startswith("SHA256:")
assert oct(clave.stat().st_mode & 0o777) == "0o600"

La siguiente celda creará el par de claves de la práctica en `$HOME/cd441-destino/ssh/`, fuera del repositorio, como indica el cuaderno original. La clave privada nunca se versiona; la celda se niega a sobrescribir una clave existente.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
DESTINO="${HOME}/cd441-destino/ssh"
mkdir -p "${DESTINO}"
if [ -e "${DESTINO}/deploy" ]; then
  echo "Ya existe ${DESTINO}/deploy; no se sobrescribe." >&2
  exit 1
fi
ssh-keygen -t ed25519 -f "${DESTINO}/deploy" -N "" -C "cd441-practica"
ssh-keygen -lf "${DESTINO}/deploy.pub"

**Salida esperada.** Una línea con la huella, de la forma `256 SHA256:<43 caracteres> cd441-practica (ED25519)`.

La siguiente celda copiará `api/deploy/` y la clave pública al *Ubuntu* y ejecutará allí `preparar_anfitrion.sh` con privilegios de administración. Las variables `ANFITRION` (dirección) y `ADMIN` (usuario administrador) se leen del entorno.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${ANFITRION:?Defina ANFITRION (dirección del Ubuntu)}" "${ADMIN:?Defina ADMIN (usuario administrador)}"
scp -r api/deploy "${HOME}/cd441-destino/ssh/deploy.pub" "${ADMIN}@${ANFITRION}:/tmp/"
ssh -t "${ADMIN}@${ANFITRION}" \
  "sudo bash /tmp/deploy/preparar_anfitrion.sh /tmp/deploy.pub deploy /opt/cd441"

**Salida esperada.** El guion termina con las líneas `== 5. Huella de la clave ed25519 de ESTE servidor`, una huella `256 SHA256:… (ED25519)` y `Listo. Configura SSH_DEPLOY_DIR=/opt/cd441 y SSH_USER=deploy en GitHub.` Es idempotente: se puede repetir.

La siguiente celda registrará un *runner* autoalojado **efímero** en el *Ubuntu*, con un usuario `runner` propio que no pertenece al grupo `docker`. Comprueba antes que el repositorio sea privado. El *token* de registro se obtiene en el equipo de trabajo, viaja por la entrada estándar de `ssh` hacia un archivo con permisos `600` y se borra al terminar.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${ANFITRION:?}" "${ADMIN:?}" "${RUNNER_VERSION:=2.332.0}"
[ "$(gh repo view --json isPrivate --jq .isPrivate)" = "true" ] || { echo "Repositorio público: no se registra un runner." >&2; exit 1; }
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
gh api --method POST "repos/${REPO}/actions/runners/registration-token" --jq .token \
  | ssh "${ADMIN}@${ANFITRION}" 'umask 077; cat > /tmp/runner-token'
ssh "${ADMIN}@${ANFITRION}" bash -s -- "${REPO}" "${RUNNER_VERSION}" <<'REMOTO'
set -euo pipefail
REPO="$1"; VERSION="$2"
TOKEN="$(cat /tmp/runner-token)"; rm -f /tmp/runner-token
id runner >/dev/null 2>&1 || sudo useradd --create-home --shell /bin/bash runner
if id -nG runner | tr ' ' '\n' | grep -qx docker; then echo "runner no debe estar en el grupo docker" >&2; exit 1; fi
sudo -H -u runner bash -s -- "${REPO}" "${VERSION}" "${TOKEN}" <<'COMO_RUNNER'
set -euo pipefail
mkdir -p "${HOME}/actions-runner" && cd "${HOME}/actions-runner"
curl -fsSL -o runner.tar.gz "https://github.com/actions/runner/releases/download/v${2}/actions-runner-linux-x64-${2}.tar.gz"
tar xzf runner.tar.gz
./config.sh --unattended --ephemeral --url "https://github.com/${1}" --token "${3}" --labels self-hosted,linux
nohup ./run.sh > runner.log 2>&1 &
COMO_RUNNER
REMOTO

**Salida esperada.** `Runner successfully added` y, en `Settings` → `Actions` → `Runners`, un *runner* en estado `Idle` con las etiquetas `self-hosted` y `Linux`. Al ser efímero, atiende **un** *job* y se da de baja; para repetir la práctica hay que registrarlo de nuevo. Si el anfitrión es alcanzable desde internet, se omite esta celda y se usa la siguiente.

La siguiente celda define la alternativa: usar un *runner* hospedado por *GitHub*. Solo es válida si el anfitrión tiene dirección pública y el puerto *SSH* abierto.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
gh variable set DEPLOY_RUNNER --env linux --body ubuntu-latest
gh variable list --env linux

## Ejercicio 2. Verificar la huella del servidor y cargar los secretos del entorno linux

Verificar la huella del servidor, cargar en el entorno `linux` los secretos `SSH_HOST`, `SSH_USER`, `SSH_PORT`, `SSH_PRIVATE_KEY` y `SSH_KNOWN_HOSTS`, y las variables `SSH_DEPLOY_DIR` y, si aplica, `DEPLOY_RUNNER`.

Verificar la huella significa comprobar que la clave pública que se guarda en `SSH_KNOWN_HOSTS` es **la del servidor**, y no la de otra máquina que se hace pasar por él. El procedimiento es obtener la clave con `ssh-keyscan` **una vez, desde el equipo de trabajo**, calcular su huella y compararla con la que el guion imprimió en el propio servidor.

La siguiente celda ensayará el procedimiento con claves generadas al vuelo: una clave hace de clave del servidor y otra de un impostor. Construye una línea de `known_hosts` con la clave pública, calcula su huella con `ssh-keygen -lf` y comprueba que coincide con la huella «impresa por el servidor» solo en el caso legítimo.

In [ ]:
import subprocess

def huella_de(archivo):
    salida = subprocess.run(["ssh-keygen", "-lf", str(archivo)], capture_output=True, text=True, check=True).stdout
    return salida.split()[1]

servidor, impostor = TEMPORAL / "servidor", TEMPORAL / "impostor"
for clave_de_prueba in (servidor, impostor):
    subprocess.run(["ssh-keygen", "-q", "-t", "ed25519", "-f", str(clave_de_prueba), "-N", ""], check=True)

tipo, material = servidor.with_suffix(".pub").read_text().split()[:2]
known_hosts = TEMPORAL / "known_hosts"
known_hosts.write_text(f"192.0.2.10 {tipo} {material}\n", encoding="utf-8")

impresa_por_el_servidor = huella_de(servidor.with_suffix(".pub"))
obtenida_con_keyscan = huella_de(known_hosts)
print("Servidor  :", impresa_por_el_servidor)
print("known_hosts:", obtenida_con_keyscan)
assert obtenida_con_keyscan == impresa_por_el_servidor
assert obtenida_con_keyscan != huella_de(impostor.with_suffix(".pub"))

La siguiente celda ejecutará el paso real `Comprobar y validar secretos y variables` de `deploy-linux-ssh.yml` con cuatro configuraciones y comprobará su comportamiento: sin secretos se omite el despliegue con éxito; con un secreto faltante falla; con valores válidos se configura; y con una dirección con caracteres peligrosos falla. Los valores son marcadores; el valor de las claves es una palabra de ejemplo.

In [ ]:
WORKFLOW = RAIZ / ".github" / "workflows" / "deploy-linux-ssh.yml"
PASO = "Comprobar y validar secretos y variables"
valido = {"SSH_HOST": "192.0.2.10", "SSH_USER": "deploy", "SSH_PORT": "22", "SSH_DEPLOY_DIR": "/opt/cd441",
          "APP_VERSION": "0" * 40, "REPOSITORIO": "Propietario/cd441",
          "SSH_PRIVATE_KEY": "marcador", "SSH_KNOWN_HOSTS": "marcador"}
base = {"REPOSITORIO": "Propietario/cd441", "APP_VERSION": "0" * 40}

vacio = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, base)
parcial = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, {**valido, "SSH_KNOWN_HOSTS": ""})
correcto = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, valido)
peligroso = u.ejecutar_paso_run(WORKFLOW, "desplegar", PASO, {**valido, "SSH_HOST": "host; reboot"})

for nombre, resultado in (("sin secretos", vacio), ("parcial", parcial), ("completo", correcto), ("peligroso", peligroso)):
    print(f"{nombre:14} código {resultado['codigo']} salidas {resultado['salidas']}")

assert (vacio["codigo"], vacio["salidas"]) == (0, {"configurado": "false"})
assert parcial["codigo"] == 1 and "SSH_KNOWN_HOSTS" in parcial["texto"]
assert (correcto["codigo"], correcto["salidas"]) == (0, {"configurado": "true"})
assert peligroso["codigo"] == 1 and "SSH_HOST no válido" in peligroso["texto"]

La siguiente celda obtendrá la clave pública del servidor con `ssh-keyscan` (solo aquí, nunca en el *pipeline*), mostrará su huella para compararla **a mano** con la que imprimió `preparar_anfitrion.sh`, y escribirá el archivo `known_hosts`.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${SSH_HOST:?}" "${SSH_PORT:=22}"
ssh-keyscan -t ed25519 -p "${SSH_PORT}" "${SSH_HOST}" > known_hosts.verificar
ssh-keygen -lf known_hosts.verificar

**Salida esperada.** Una línea `256 SHA256:… (ED25519)` que debe ser **idéntica** a la huella que imprimió el guion de preparación en el servidor. Si difiere, no se continúa: puede haber una máquina intermedia. Cuando coincide, el archivo `known_hosts.verificar` es el valor de `SSH_KNOWN_HOSTS`. Con un puerto distinto de 22, `ssh-keyscan` escribe la entrada con el formato `[host]:puerto`.

La siguiente celda cargará los cinco secretos y las dos variables en el entorno `linux`. La clave privada y el `known_hosts` se leen de archivos con la entrada estándar; el resto, de variables de entorno. Ningún valor se imprime.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${SSH_HOST:?}" "${SSH_USER:=deploy}" "${SSH_PORT:=22}" "${SSH_DEPLOY_DIR:=/opt/cd441}"
printf '%s' "${SSH_HOST}" | gh secret set SSH_HOST --env linux
printf '%s' "${SSH_USER}" | gh secret set SSH_USER --env linux
printf '%s' "${SSH_PORT}" | gh secret set SSH_PORT --env linux
gh secret set SSH_PRIVATE_KEY --env linux < "${HOME}/cd441-destino/ssh/deploy"
gh secret set SSH_KNOWN_HOSTS --env linux < known_hosts.verificar
gh variable set SSH_DEPLOY_DIR --env linux --body "${SSH_DEPLOY_DIR}"
rm -f known_hosts.verificar
gh secret list --env linux
gh variable list --env linux

**Salida esperada.** Cinco secretos (`SSH_HOST`, `SSH_KNOWN_HOSTS`, `SSH_PORT`, `SSH_PRIVATE_KEY`, `SSH_USER`) con su fecha y las variables `SSH_DEPLOY_DIR` y, si se definió, `DEPLOY_RUNNER`. Los valores de los secretos no se pueden leer.

## Ejercicio 3. Desplegar con el workflow y comprobar GET /health

Tras un *CI* verde en `main` que haya publicado la imagen en *GHCR*, disparar el *workflow* `Despliegue en Linux por SSH` y comprobar que `GET /health` responde con `"almacen":"postgresql"` y la versión del *SHA*.

La siguiente celda calculará con las reglas del *workflow* la imagen que se descargará en el servidor y comprobará con `assert` que usa la etiqueta del *SHA* en minúsculas y que `APP_VERSION` cumple el formato de 40 caracteres hexadecimales que exige el paso de validación.

In [ ]:
import re

SHA = "0123456789abcdef0123456789abcdef01234567"   # ejemplo
imagen = f"ghcr.io/{'Propietario/cd441'.lower()}/api:{SHA}"
print(imagen)
assert re.fullmatch(r"[0-9a-f]{40}", SHA)
assert imagen == "ghcr.io/propietario/cd441/api:" + SHA

texto = WORKFLOW.read_text(encoding="utf-8")
assert 'API_IMAGE=ghcr.io/${REPOSITORIO,,}/api:${APP_VERSION}' in texto
assert "APP_VERSION: ${{ github.sha }}" in texto

La siguiente celda disparará el *workflow*, esperará su resultado y mostrará el cuerpo de `/health` que el propio *workflow* imprimió en su registro.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
gh workflow run "Despliegue en Linux por SSH" --ref main
sleep 10
ID="$(gh run list --workflow "Despliegue en Linux por SSH" --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run watch "${ID}" --exit-status
gh run view "${ID}" --log | grep -E '"almacen"'
git rev-parse HEAD

**Salida esperada.** Una línea `{"almacen":"postgresql","status":"ok","version":"<SHA>"}` y, a continuación, el mismo *SHA* de 40 caracteres que imprime `git rev-parse HEAD`. La salida de ese tipo que muestra el cuaderno original es ilustrativa: el propio cuaderno indica que no se verificó contra un anfitrión real. Esta solución tampoco se ha verificado contra uno.

## Ejercicio 4. Comprobar la idempotencia y la persistencia de los datos

Volver a disparar el *workflow* sin cambiar el código y verificar que el resultado es el mismo (idempotencia). Comprobar que los datos sobreviven a `systemctl restart cd441-api` y a `systemctl restart postgresql`.

El enunciado no dice qué datos crear ni cómo verificarlos. La solución crea una tarea con `POST /tareas` en el servidor, reinicia los dos servicios y comprueba con `GET /tareas` que la tarea sigue ahí con el mismo identificador. La *API* abre una conexión a *PostgreSQL* en cada petición, por lo que se espera que se recupere sola tras reiniciar la base.

La siguiente celda comprobará en los archivos que la unidad `systemd` y el *workflow* ejecutan el mismo comando de *Compose* (`up -d --wait`), que es lo que hace idempotente al despliegue: si nada cambió, *Compose* no recrea el contenedor.

In [ ]:
unidad = (RAIZ / "api" / "deploy" / "cd441-api.service").read_text(encoding="utf-8")
comando = "docker compose -f api/deploy/compose.yaml --env-file .env up -d --wait"
print([l for l in unidad.splitlines() if l.startswith("ExecStart")])
assert comando in unidad
assert comando in WORKFLOW.read_text(encoding="utf-8")

La siguiente celda volverá a disparar el *workflow* sin cambios y comparará, en el servidor, el identificador y la hora de inicio del contenedor antes y después. Si coinciden, la repetición no recreó el contenedor.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${ANFITRION:?}" "${ADMIN:?}" "${SSH_DEPLOY_DIR:=/opt/cd441}"
estado() {
  ssh "${ADMIN}@${ANFITRION}" "cd '${SSH_DEPLOY_DIR}' && docker inspect --format '{{.Id}} {{.State.StartedAt}}' \
    \$(docker compose -f api/deploy/compose.yaml --env-file .env ps -q api)"
}
ANTES="$(estado)"
gh workflow run "Despliegue en Linux por SSH" --ref main
sleep 10
gh run watch "$(gh run list --workflow 'Despliegue en Linux por SSH' --limit 1 --json databaseId --jq '.[0].databaseId')" --exit-status
DESPUES="$(estado)"
[ "${ANTES}" = "${DESPUES}" ] && echo "IDEMPOTENTE: el contenedor no se recreó" || echo "El contenedor cambió"

**Salida esperada.** `IDEMPOTENTE: el contenedor no se recreó`. En el registro del *workflow*, `docker compose up` indica `Running` (o `Healthy`) en lugar de `Recreated`.

La siguiente celda creará una tarea en el servidor, reiniciará `cd441-api` y `postgresql`, y comprobará que la tarea persiste y que `/health` vuelve a informar `postgresql`. Se ejecuta en el *Ubuntu* por *SSH*; la *API* escucha solo en `127.0.0.1:8080`, por lo que las peticiones se hacen desde el propio servidor.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
: "${ANFITRION:?}" "${ADMIN:?}"
ssh "${ADMIN}@${ANFITRION}" bash -s <<'REMOTO'
set -euo pipefail
API=http://127.0.0.1:8080
curl -fsS -X POST "${API}/tareas" -H 'Content-Type: application/json' -d '{"titulo": "prueba de persistencia"}'; echo
sudo systemctl restart cd441-api
curl -fsS --retry 10 --retry-all-errors --retry-delay 2 "${API}/health"; echo
sudo systemctl restart postgresql
curl -fsS --retry 10 --retry-all-errors --retry-delay 2 "${API}/tareas"; echo
REMOTO

**Salida esperada.** La primera línea devuelve la tarea creada, con `"id"` y `"titulo": "prueba de persistencia"` (código 201). Después de reiniciar `cd441-api`, `/health` informa `"almacen":"postgresql"` y `"status":"ok"`. Después de reiniciar `postgresql`, `GET /tareas` devuelve una lista que incluye la misma tarea con el mismo `id`. Un fallo breve durante el reinicio de la base es normal; `--retry` lo absorbe.

## Ejercicio 5. Localizar la guarda, el known_hosts y el borrado de la clave

Localizar en el *workflow* la guarda que impide un *runner* autoalojado en un repositorio público, el uso de `SSH_KNOWN_HOSTS` y el paso que borra la clave del *runner*.

La siguiente celda localizará los tres elementos y los comprobará: el primer paso con su condición, el uso de `StrictHostKeyChecking=yes` con el `known_hosts` fijado (y la ausencia de `ssh-keyscan` en líneas ejecutables), y el último paso con `if: always()` que borra el directorio de la clave.

In [ ]:
wf = u.cargar_workflow(WORKFLOW)
pasos_ssh = u.pasos(wf, "desplegar")

guarda = pasos_ssh[0]
print("Guarda       :", guarda["name"])
print("  condición  :", guarda["if"])
assert "runner.environment == 'self-hosted'" in guarda["if"] and "github.event.repository.private != true" in guarda["if"]
assert "exit 1" in guarda["run"]

texto = WORKFLOW.read_text(encoding="utf-8")
ejecutables = [l for l in texto.splitlines() if not l.lstrip().startswith("#")]
assert any("StrictHostKeyChecking=yes" in l for l in ejecutables)
assert any("UserKnownHostsFile=" in l for l in ejecutables)
assert not any("ssh-keyscan" in l for l in ejecutables)
preparar = next(p for p in pasos_ssh if p.get("name", "").startswith("Preparar clave y known_hosts"))
print("known_hosts  :", preparar["name"], "-> secreto", preparar["env"]["SSH_KNOWN_HOSTS"])
assert preparar["env"]["SSH_KNOWN_HOSTS"] == "${{ secrets.SSH_KNOWN_HOSTS }}"

ultimo = pasos_ssh[-1]
print("Borrado      :", ultimo["name"], "| if:", ultimo["if"])
assert ultimo["if"] == "always()" and "rm -rf" in ultimo["run"] and "RUNNER_TEMP" in ultimo["run"]

## Ejercicio 6. Dar de baja el destino y retirar los secretos (complemento)

El cuaderno 17 no incluye la limpieza. Al terminar la práctica, deben retirarse los secretos y la clave de la práctica, y darse de baja el *runner*, aunque la máquina sea desechable.

La siguiente celda borrará los secretos y las variables del entorno `linux`, la clave local de la práctica y la clave autorizada del usuario de despliegue en el *Ubuntu*. Si el *runner* sigue registrado, se da de baja como en el cuaderno 10.

In [ ]:
%%bash
set -uo pipefail
cd "$(git rev-parse --show-toplevel)"
for NOMBRE in SSH_HOST SSH_USER SSH_PORT SSH_PRIVATE_KEY SSH_KNOWN_HOSTS; do gh secret delete "${NOMBRE}" --env linux; done
for NOMBRE in SSH_DEPLOY_DIR DEPLOY_RUNNER; do gh variable delete "${NOMBRE}" --env linux; done
rm -rf "${HOME}/cd441-destino"
ssh "${ADMIN:?}@${ANFITRION:?}" "sudo rm -f /home/deploy/.ssh/authorized_keys"
gh secret list --env linux

**Salida esperada.** Una lista de secretos vacía. Si la máquina era desechable, lo más limpio es destruirla junto con su disco.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2027.</p>